In [1]:
# Mercedes-Benz Test Time Prediction
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import xgboost as xgb


In [2]:
# Load Data
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (4209, 378)
Test shape: (4209, 377)


In [3]:
# Remove zero variance columns
zero_var_cols = train_df.columns[train_df.nunique() == 1]
train_df.drop(columns=zero_var_cols, inplace=True)
test_df.drop(columns=zero_var_cols, inplace=True)

print("Removed zero variance columns:", list(zero_var_cols))

Removed zero variance columns: ['X11', 'X93', 'X107', 'X233', 'X235', 'X268', 'X289', 'X290', 'X293', 'X297', 'X330', 'X347']


In [4]:
# Check null values
print("\nNull values in train:\n", train_df.isnull().sum())
print("\nNull values in test:\n", test_df.isnull().sum())


Null values in train:
 ID      0
y       0
X0      0
X1      0
X2      0
       ..
X380    0
X382    0
X383    0
X384    0
X385    0
Length: 366, dtype: int64

Null values in test:
 ID      0
X0      0
X1      0
X2      0
X3      0
       ..
X380    0
X382    0
X383    0
X384    0
X385    0
Length: 365, dtype: int64


In [5]:
# Check unique values
print("\nUnique values in train:\n", train_df.nunique())
print("\nUnique values in test:\n", test_df.nunique())


Unique values in train:
 ID      4209
y       2545
X0        47
X1        27
X2        44
        ... 
X380       2
X382       2
X383       2
X384       2
X385       2
Length: 366, dtype: int64

Unique values in test:
 ID      4209
X0        49
X1        27
X2        45
X3         7
        ... 
X380       2
X382       2
X383       2
X384       2
X385       2
Length: 365, dtype: int64


In [6]:
# Separate target
y = train_df["y"]
train_df.drop(["y"], axis=1, inplace=True)

# Save test IDs
test_ids = test_df["ID"]

# Drop ID column
train_df.drop(["ID"], axis=1, inplace=True)
test_df.drop(["ID"], axis=1, inplace=True)

In [7]:
# Label Encoding
for col in train_df.columns:
    if train_df[col].dtype == 'object':
        le = LabelEncoder()
        combined = pd.concat([train_df[col], test_df[col]], axis=0)
        le.fit(combined.astype(str))

        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))

# Dimensionality Reduction (PCA)
n_components = 50  # you can tune this
pca = PCA(n_components=n_components, random_state=42)

train_pca = pca.fit_transform(train_df)
test_pca = pca.transform(test_df)

# Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    train_pca, y, test_size=0.2, random_state=42
)

# XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# Validation Performance
y_pred_val = model.predict(X_val)
print("\nValidation R2 Score:", r2_score(y_val, y_pred_val))

# Predict Test Set
test_predictions = model.predict(test_pca)

# Save Submission
submission = pd.DataFrame({
    "ID": test_ids,
    "y": test_predictions
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission file created: submission.csv")


Validation R2 Score: 0.4918655193100996

Submission file created: submission.csv
